# L7c: Ordinary Least Squares
To estimate the parameters of a linear model, we use __ordinary least squares (OLS)__ and explore its mathematical foundations through both direct and SVD-based solutions.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Estimate linear model parameters with ordinary least squares:__ Ordinary least squares chooses the parameters of a linear model that minimize the sum of squared differences between the observed and predicted outputs. We'll derive the estimate when the data matrix has more observations than parameters, and the smallest-norm estimate when it has fewer.
> * __Compute the estimate with the singular value decomposition:__ The singular value decomposition (SVD) writes the least-squares estimate as a weighted sum of the right singular vectors of the data matrix. We'll compare this approach with solving the normal equations directly, in terms of numerical stability and computational cost.
> * __Quantify the uncertainty in the estimated parameters:__ Because the observations contain random error, the estimated parameters are random variables with their own variance. We'll estimate the error variance from the residuals, compute a standard error for each parameter, and build confidence intervals from Student's t-distribution.

In the L7d lab, we'll apply these ideas to a linear model of housing prices. Let's get started!

___

## Linear models for continuous prediction tasks
A linear model describes a continuous output, such as a price or a temperature, as a weighted sum of input features plus a random error. The weights are the model's parameters, and we estimate them from data.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-LinearRegressionModel-Schematic/Fig-LinearRegressionModel-Schematic.svg" width="640" alt="Red observations scattered around a straight model line, with the response on the vertical axis and the feature on the horizontal axis. Dashed vertical segments connect each observation to the line. For observation i, guides mark x_i and y_i on the axes, an open marker on the line marks the fitted value x-hat_i transpose theta-hat, and a brace labels the residual r_i equals y_i minus x-hat_i transpose theta-hat."/>
    </center>
</div>

__Observations, model, and residuals.__ Each red point is an observation, and the line is the model's prediction. Each dashed segment is a residual, the vertical distance between an observed output and the prediction at the same input. Least squares chooses the parameters that make the sum of the squared residuals as small as possible. The labels use the augmented features $\hat{\mathbf{x}}_{i}$ and the estimated parameters $\hat{\mathbf{\theta}}$, which we define below.

Let's write the model for a single observation, then for the whole dataset.

Suppose there exists a dataset $\mathcal{D} = \left\{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\right\}$ with $n$ training (labeled) examples, where $\mathbf{x}_{i}\in\mathbb{R}^{m}$ is an $m$-dimensional vector of features (independent input variables) and $y_{i}\in\mathbb{R}$ denotes a scalar response variable (dependent variable). Then, a __linear model__ for the dataset $\mathcal{D}$ is given (in index-form) by:
$$
\begin{equation*}
y_{i} = \hat{\mathbf{x}}_{i}^{\top}\,\mathbf{\theta} + \epsilon_{i}\qquad{i=1,2,\dots,n}
\end{equation*}
$$
where the augmented features are $\hat{\mathbf{x}}_{i}^{\top}=\left(x_{i1},x_{i2},\dots,x_{im},1\right)$ (we've added an extra `1` to each feature vector to account for the intercept (bias) term),
the unknown parameters are represented by the $\mathbf{\theta}\in\mathbb{R}^{p}$ vector (where $p=m+1$), and $\epsilon_{i}\in\mathbb{R}$ is the unobserved random error for response $i$, i.e., the component of the target that is _not_ explained by the linear model.

We can rewrite the linear regression model in matrix-vector form as:
$$
\begin{equation*}
\mathbf{y} = \hat{\mathbf{X}}\;\mathbf{\theta} + \mathbf{\epsilon}
\end{equation*}
$$
where $\hat{\mathbf{X}}$ is an $n\times{p}$ matrix with the augmented features $\hat{\mathbf{x}}_{i}^{\top}$ on the rows, the target (output) vector $\mathbf{y}$ is an $n\times{1}$ column vector with entries $y_{i}$, and the error vector $\mathbf{\epsilon}$ is an $n\times{1}$ column vector with entries $\epsilon_{i}$. The challenge of linear regression is to estimate the unknown parameters $\mathbf{\theta}$ from the dataset $\mathcal{D}$ by minimizing an appropriate loss function, typically the sum of squared errors.

> __Key insight:__ A linear model only needs to be linear in the parameters, not in the features. For example, with a single scalar input $x$, we could have polynomial features in the data matrix $\hat{\mathbf{X}}$ such as $1,x,x^{2},x^{3},\dots$. This would still be a linear regression problem because the model remains linear in the parameters $\mathbf{\theta}$.

___


## Ordinary least squares
The form of the ordinary least squares (OLS) estimate depends on the shape of the data matrix: whether we have more observations than parameters, or fewer.

### Overdetermined data matrix without regularization
Suppose you have a data matrix $\hat{\mathbf{X}}\in\mathbb{R}^{n\times{p}}$ that is __overdetermined__, i.e., $n > p$ (more observations than parameters), and an error model $\mathbf{\epsilon}\sim\mathcal{N}(\mathbf{0},\sigma^{2}\;\mathbf{I})$ that follows [a normal distribution](https://en.wikipedia.org/wiki/Normal_distribution) with a mean of zero and variance $\sigma^{2}$. We estimate the model parameters by minimizing the sum of squared errors between the model's estimated outputs and the observed outputs:
$$
\begin{align*}
\hat{\mathbf{\theta}} = \arg\min_{\mathbf{\theta}} \frac{1}{2}\;\lVert~\mathbf{y} - \hat{\mathbf{X}}\;\mathbf{\theta}~\rVert^{2}_{2}
\end{align*}
$$
where $\lVert\star\rVert^{2}_{2}$ is the square of the [L2 vector norm](https://en.wikipedia.org/wiki/Norm_(mathematics)#Euclidean_norm), and $\hat{\mathbf{\theta}}\in\mathbb{R}^{p}$ is the estimated parameter vector. Setting the gradient of this objective with respect to $\mathbf{\theta}$ to zero gives the __normal equations__:
$$
\begin{align*}
\hat{\mathbf{X}}^{\top}\left(\mathbf{y} - \hat{\mathbf{X}}\;\hat{\mathbf{\theta}}\right) = \mathbf{0}
\end{align*}
$$
The residual $\mathbf{y} - \hat{\mathbf{X}}\hat{\mathbf{\theta}}$ is orthogonal to every column of $\hat{\mathbf{X}}$, so the fitted values $\hat{\mathbf{X}}\hat{\mathbf{\theta}}$ are the orthogonal projection of $\mathbf{y}$ onto the column space of $\hat{\mathbf{X}}$. When $\hat{\mathbf{X}}$ has full column rank (i.e., its rank is $r(\hat{\mathbf{X}}) = p$ and $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$ is invertible), the normal equations have the analytical solution:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
\end{align*}
$$

We can also express the estimated parameters in terms of the true parameters and the error model $\mathbf{\epsilon}$:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y} \\
\hat{\mathbf{\theta}}&= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}(\hat{\mathbf{X}}\;\mathbf{\theta} + \mathbf{\epsilon}) \\
\hat{\mathbf{\theta}} &= \underbrace{\left[\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right]}_{= \mathbf{I}}\;\mathbf{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon}\\
\hat{\mathbf{\theta}} &= \mathbf{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon}\quad\blacksquare\\
\end{align*}
$$

> __Key insights:__
>
> * __Unbiased estimator:__ The errors have mean zero, so taking the expected value of the last line gives $\mathbb{E}[\hat{\mathbf{\theta}}] = \mathbf{\theta}$. OLS is an unbiased estimator: on average, we get the right answer.
> * __Parameter uncertainty:__ Since $\mathbf{\epsilon}$ is a random vector, $\hat{\mathbf{\theta}}$ is also a random vector with its own distribution. We quantify this uncertainty with standard errors and confidence intervals later in this lecture.
> * __Connection to Bayesian inference:__ The decomposition $\hat{\mathbf{\theta}} = \mathbf{\theta} + (\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon}$ shows that even in frequentist regression, parameter estimates have distributions. This provides a conceptual bridge toward [Bayesian linear regression](https://en.wikipedia.org/wiki/Bayesian_linear_regression), where the parameters themselves are treated as random variables with prior distributions.

### Underdetermined data matrix without regularization
Next, let's assume the data matrix $\hat{\mathbf{X}}$ is __underdetermined__, i.e., $n < p$ (fewer observations than parameters), with full row rank, $r(\hat{\mathbf{X}}) = n$, and the error model $\mathbf{\epsilon}\sim\mathcal{N}(\mathbf{0},\sigma^{2}\mathbf{I})$. Then infinitely many parameter vectors satisfy $\hat{\mathbf{X}}\mathbf{\theta} = \mathbf{y}$ exactly, and each of them makes the sum of squared errors zero. Among these least-squares solutions, we choose the _smallest_ one:
$$
\begin{align*}
\text{minimize}~&  ||\,\mathbf{\theta}\,||^{2}_{2} \\
\text{subject to} & \, \hat{\mathbf{X}}\;\mathbf{\theta} = \mathbf{y}
\end{align*}
$$
Because $\hat{\mathbf{X}}$ has full row rank, the matrix $\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}$ is invertible and the least-norm solution of the unknown parameter vector is given by:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\mathbf{y}
\end{align*}
$$
We can express the solution in terms of the error model $\mathbf{\epsilon}$:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\mathbf{y}\\
\hat{\mathbf{\theta}} &= \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;(\hat{\mathbf{X}}\;\mathbf{\theta} + \mathbf{\epsilon})\\
\hat{\mathbf{\theta}} &= \underbrace{\hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\hat{\mathbf{X}}}_{\text{Projection}\;\mathbf{P}}\;\mathbf{\theta} + \hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\mathbf{\epsilon}\\
\hat{\mathbf{\theta}} &= \mathbf{P}\;\mathbf{\theta} +\hat{\mathbf{X}}^{\top}\left(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top}\right)^{-1}\;\mathbf{\epsilon}\quad\blacksquare\\
\end{align*}
$$

> __Projection interpretation:__ The matrix $\mathbf{P} = \hat{\mathbf{X}}^{\top}(\hat{\mathbf{X}}\hat{\mathbf{X}}^{\top})^{-1}\hat{\mathbf{X}}$ is the orthogonal projection onto the row space of $\hat{\mathbf{X}}$, the span of the augmented feature vectors $\hat{\mathbf{x}}_{i}$. Because $\mathbf{P} \neq \mathbf{I}$ when $n < p$, the expected estimate is $\mathbb{E}[\hat{\mathbf{\theta}}] = \mathbf{P}\mathbf{\theta}$, which is biased unless $\mathbf{\theta}$ already lies in that subspace.

__Important questions:__ A few key considerations arise here.

* _Why do we select the smallest $\hat{\mathbf{\theta}}$?_ Every exact solution is the least-norm solution $\hat{\mathbf{\theta}}$ plus a vector $\mathbf{z}$ in the null space of $\hat{\mathbf{X}}$, i.e., $\hat{\mathbf{X}}\mathbf{z} = \mathbf{0}$. The least-norm solution lies in the row space, which is orthogonal to the null space, so $\lVert\hat{\mathbf{\theta}} + \mathbf{z}\rVert_{2}^{2} = \lVert\hat{\mathbf{\theta}}\rVert_{2}^{2} + \lVert\mathbf{z}\rVert_{2}^{2}$. Adding $\mathbf{z}$ makes the parameter vector longer without changing the predictions at the observed data, so the least-norm solution keeps only the part of $\mathbf{\theta}$ that the data determine. This is the same preference for small parameters that L2 regularization builds into the loss function.
* _What is regularization?_ Regularization is a method used to prevent overfitting in machine learning models by adding a penalty to the loss function. In linear regression, regularization helps manage the model's complexity and enhances its ability to generalize to new data. Common techniques include Lasso (L1 regularization) and Ridge (L2 regularization).

Now let's explore an alternative computational approach using the singular value decomposition, which provides better numerical stability and additional insight into the structure of our data.

___

## SVD solution of the least-squares problem
The singular value decomposition (SVD) of the $n\times{p}$ data matrix $\hat{\mathbf{X}}$ is given by:
$$
\begin{equation*}
\hat{\mathbf{X}} = \mathbf{U}\;\mathbf{S}\;\mathbf{V}^{\top}
\end{equation*}
$$
where $\mathbf{U} \in \mathbb{R}^{n \times n}$ is an orthogonal matrix, $\mathbf{S} \in \mathbb{R}^{n \times p}$ is a rectangular diagonal matrix with the singular values $\sigma_{1}\geq\cdots\geq\sigma_{r}>0$ on its first $r=r(\hat{\mathbf{X}})$ diagonal entries and zeros elsewhere, and $\mathbf{V} \in \mathbb{R}^{p \times p}$ is an orthogonal matrix. This is the SVD notation from the L7a lecture. The least-squares estimate of the unknown parameter vector $\mathbf{\theta}$ is given by:
$$
\begin{equation*}
\hat{\mathbf{\theta}} = \mathbf{V}\;\mathbf{S}^{\dagger}\;\mathbf{U}^{\top}\;\mathbf{y}
\end{equation*}
$$
where $\mathbf{S}^{\dagger}\in\mathbb{R}^{p\times{n}}$ is the Moore-Penrose pseudoinverse of $\mathbf{S}$: it transposes $\mathbf{S}$ and replaces each nonzero singular value $\sigma_{i}$ with $1/\sigma_{i}$. For practical computation, this can be written in index notation as:
$$
\boxed{
\begin{equation*}
\hat{\mathbf{\theta}} = \sum_{i=1}^{r(\hat{\mathbf{X}})}\left(\frac{\mathbf{u}_{i}^{\top}\mathbf{y}}{\sigma_{i}}\right)\mathbf{v}_{i}\quad\blacksquare
\end{equation*}}
$$
where $r(\hat{\mathbf{X}})\leq\min(n,p)$ is the rank of the data matrix $\hat{\mathbf{X}}$ (the number of nonzero singular values), $\mathbf{u}_{i}$ and $\mathbf{v}_{i}$ are the $i$-th columns of $\mathbf{U}$ and $\mathbf{V}$, respectively, and $\sigma_{i}>0$ is the $i$-th singular value. The same formula covers both cases above: it gives the OLS estimate when $\hat{\mathbf{X}}$ has full column rank, and the least-norm estimate when $\hat{\mathbf{X}}$ has full row rank.

This is just a weighted sum of the right singular vectors $\mathbf{v}_{i}$, where the weights depend on how well the data $\mathbf{y}$ aligns with the left singular vectors $\mathbf{u}_{i}$ (the dot products $\mathbf{u}_{i}^{\top}\mathbf{y}$, see the [dot product schematic](figs/Fig-InnerProduct-NeedToRedrawThis.png)) and are scaled by the inverse of the singular values $\sigma_{i}$.

> __Key insights:__
>
> * __Mode contribution decomposition:__ Each term in the sum is one mode. The vectors $\mathbf{v}_{i}$ are orthogonal directions in parameter space and the $\mathbf{u}_{i}$ are the corresponding directions in data space, so each term projects $\mathbf{y}$ onto $\mathbf{u}_{i}$ and maps the result onto $\mathbf{v}_{i}$.
> * __Noise amplification diagnosis:__ The factor $1/\sigma_{i}$ shows which modes amplify noise. When $\sigma_i$ is very small, measurement errors in $\mathbf{y}$ are magnified along $\mathbf{v}_{i}$, which identifies unreliable components of the solution and motivates regularization techniques that dampen the contributions of small singular values.

### SVD versus direct methods
The SVD route is an alternative to solving the normal equations directly, as we did for the overdetermined case. Why choose one over the other? Here are some considerations:

> __SVD versus direct method?__
>
> * __Rank-deficient matrices:__ When $\hat{\mathbf{X}}$ doesn't have full column rank, $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$ is singular and cannot be inverted. The SVD handles this case gracefully: the sum skips the zero singular values and returns the minimum-norm least-squares solution. In floating point, singular values below a small tolerance are treated as zero.
>
> * __Ill-conditioned problems:__ The condition number $\kappa$ of a matrix is the ratio of its largest to its smallest singular value. When $\hat{\mathbf{X}}$ has full column rank, $\kappa(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}) = \kappa(\hat{\mathbf{X}})^{2}$, so forming $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$ squares the condition number and amplifies rounding errors when $\hat{\mathbf{X}}$ is ill-conditioned. The SVD factors $\hat{\mathbf{X}}$ directly and avoids the squaring.

__Computational trade-offs:__ The thin SVD of $\hat{\mathbf{X}}$, which keeps $\min(n,p)$ singular vectors on each side, costs $O(\min(np^2, n^2p))$ operations. The direct method costs $O(np^2)$ to form $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$ and $O(p^3)$ to solve the resulting $p\times{p}$ system. For tall, thin matrices ($n \gg p$), both scale as $O(np^2)$, but the direct method has a smaller constant and is often faster. The SVD provides better numerical properties when stability matters more than speed.

Julia's [backslash operator](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#Base.:%5C%5C-Tuple{AbstractMatrix,%20AbstractVecOrMat}) takes a middle path. For a rectangular matrix `X`, the expression `X \ y` computes the least-squares solution from a factorization of `X` itself (a pivoted QR factorization) and never forms `X' * X`.

___

## Understanding the error model
The error model $\mathbf{\epsilon}$ captures the randomness in our observations that cannot be explained by the linear relationship in the model. In linear regression, we typically assume:
$$\begin{align*}
\mathbf{\epsilon} &\sim \mathcal{N}(\mathbf{0},\sigma^{2}\;\mathbf{I})
\end{align*}$$
This means each error term $\epsilon_i$ is independent, normally distributed with mean zero and constant variance $\sigma^2$.

> __Normality assumption:__ The normality assumption gives the estimated parameters exact distributions, which allows us to construct confidence intervals, perform hypothesis tests, and quantify uncertainty. Under the weaker conditions of zero-mean, uncorrelated, constant-variance errors, OLS gives us the Best Linear Unbiased Estimator (BLUE).

> __Uncorrelated errors:__ Our error model also implies that the errors for different observations are uncorrelated, i.e., $\text{Cov}(\epsilon_i, \epsilon_j) = 0$ for $i \neq j$. We can see this from the covariance matrix $\sigma^{2}\;\mathbf{I}$, which is diagonal with $\sigma^2$ on the diagonal and zeros elsewhere. The parameter variance we derive below relies on this structure; with correlated errors, the standard errors and confidence intervals built from it would be wrong.

__Reality check:__ While the normality assumption may not always hold in practice, many results remain approximately valid thanks to the Central Limit Theorem; parameter estimates are often approximately normal for large samples, even when individual errors are not.

For the rest of this lecture, we return to the overdetermined case: $\hat{\mathbf{X}}$ has more observations than parameters and full column rank, and we treat it as fixed, so all of the randomness in $\hat{\mathbf{\theta}}$ comes from $\mathbf{\epsilon}$.

### Estimating the error variance
Since the __true__ variance $\sigma^2$ is unknown, we estimate it from the residuals $\mathbf{r} = \mathbf{y} - \hat{\mathbf{X}}\hat{\mathbf{\theta}}$ as:
$$\begin{align*}
\hat{\sigma}^{2} &= \frac{\lVert~\mathbf{r}~\rVert^{2}_{2}}{n-p} = \frac{1}{n-p}\sum_{i=1}^{n}r_i^2
\end{align*}$$
where $n$ is the number of observations, $p$ is the number of parameters, and $r_i = y_i - \hat{\mathbf{x}}_i^{\top}\hat{\mathbf{\theta}}$ is the $i$-th residual, i.e., the difference between the observed and predicted value for observation $i$.

> __Key insight:__ We divide by $(n-p)$ instead of $n$ to account for the degrees of freedom "used up" by estimating $p$ parameters. This correction makes $\hat{\sigma}^2$ an unbiased estimator of the true variance. It is also why we need $n > p$: with full row rank and $n \leq p$, the fit passes through every observation, so the residuals are zero and carry no information about $\sigma^{2}$.

### Parameter uncertainty (overdetermined case without regularization)
With our estimate of the variance $\hat{\sigma}^2$, we can now quantify the uncertainty in our parameter estimates $\text{Var}(\hat{\mathbf{\theta}})$. Starting from the decomposition we derived for the overdetermined case, the variance of the estimated parameters is given by:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \mathbf{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon}\\
\text{Var}(\hat{\mathbf{\theta}}) &= \text{Var}\left(\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon}\right)\quad\text{(since $\mathbf{\theta}$ is constant)}\\
\end{align*}
$$
For a random vector $\mathbf{A}\mathbf{z}$, where $\mathbf{z}$ is a random vector and $\mathbf{A}$ is a constant matrix, the variance is given by:
$$
\text{Var}(\mathbf{A}\mathbf{z}) = \mathbf{A}\;\text{Var}(\mathbf{z})\;\mathbf{A}^{\top}
$$
We assumed in our error model that $\text{Var}(\mathbf{\epsilon}) = \sigma^{2}\;\mathbf{I}$. Therefore:
$$
\begin{align*}
\text{Var}(\hat{\mathbf{\theta}}) &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\;\text{Var}(\mathbf{\epsilon})\;\hat{\mathbf{X}}\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\\
&= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\;(\sigma^{2}\;\mathbf{I})\;\hat{\mathbf{X}}\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\\
&= \sigma^{2}\;\underbrace{\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}}_{=\;\mathbf{I}}\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\\
&= \sigma^{2}\;\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\quad\blacksquare\\
\end{align*}
$$
Finally, the __standard error__ $\mathrm{SE}(\hat{\theta}_j)$ is the estimated standard deviation of the parameter estimate $\hat{\theta}_j$. We take the square root of the $j$-th diagonal element of the covariance matrix, with the unknown $\sigma^{2}$ replaced by its estimate $\hat{\sigma}^{2}$:
$$
\begin{align*}
  \mathrm{SE}(\hat{\theta}_j) &= \sqrt{\;\hat{\sigma}^2\;\bigl[(\hat{\mathbf{X}}^\top\hat{\mathbf{X}})^{-1}\bigr]_{jj}\,}
\end{align*}
$$
So why is this super cool? Because there is a ton of stuff that we can do with standard errors!

> Standard errors are __essential__ for:
> 
> * __Confidence intervals:__ For large samples, $\hat{\theta}_j \pm 1.96 \cdot \mathrm{SE}(\hat{\theta}_j)$ is an approximate 95% confidence interval for $\theta_j$, where 1.96 comes from the standard normal distribution. For finite samples, we use the t-distribution, as shown below.
> * __Hypothesis testing:__ To test whether $\theta_j = 0$ (is feature $j$ significant?), we compare the t-statistic $t_j = \hat{\theta}_j/{\mathrm{SE}(\hat{\theta}_j)}$ with a t-distribution with $n-p$ degrees of freedom. The p-value is the probability of a value at least this extreme if $\theta_j$ is actually 0.
> * __Prediction intervals:__ A prediction interval adds a margin of error around the prediction for a new observation. The margin combines the parameter uncertainty, through the full covariance matrix $\hat{\sigma}^{2}(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}$, with the error variance $\hat{\sigma}^{2}$ of the new observation.

### A note about confidence intervals
A __confidence interval__ gives a range of values that likely contains the true parameter value (which we don't know). For example, a 95% confidence interval means that if we were to repeat the experiment many times and compute the confidence interval each time, approximately 95% of those intervals would contain the true parameter value.

Let's dig in a bit deeper. Start with the studentized statistic $T_j$ for the __true__ parameter value $\theta_j$ (unknown), using the standard error from above:
$$
\begin{align*}
T_j \;&=\; \frac{\hat{\theta}_j-\theta_j}{\mathrm{SE}(\hat{\theta}_j)}
\end{align*}
$$
Under the normal-error model with constant variance (homoskedastic errors) and the $\hat\sigma^2$ estimate we developed above, the distribution of $T_j$ is a Student's $t$ with:
$$
\begin{align*}
T_j & \sim t_{\nu},\qquad \nu=n-p
\end{align*}
$$
where $n$ is the number of observations, $p$ is the number of parameters, and $\nu=n-p$ is the degrees of freedom.
Let $c=t_{1-\alpha/2,\nu}$ where $\alpha$ is the significance level (e.g., $\alpha=0.05$ for a 95% CI) and $t_{1-\alpha/2,\nu}$ is the $(1-\alpha/2)$ quantile of the $t$-distribution with $\nu$ degrees of freedom.
Then (given $\mathrm{SE}(\hat\theta_j)>0$):
$$
\Pr\!\big(-c \le T_j \le c\big)=1-\alpha
\quad\Longleftrightarrow\quad
\Pr\!\left(-c \le \frac{\hat\theta_j-\theta_j}{\mathrm{SE}(\hat\theta_j)} \le c\right)=1-\alpha.
$$
Now we invert the inequality to get bounds on $\theta_j$ instead of $T_j$: After some algebraic manipulation, we get:
$$
\Pr\!\big(\ \hat\theta_j - c\;\mathrm{SE}(\hat\theta_j) \;\le\; \theta_j \;\le\; \hat\theta_j + c\;\mathrm{SE}(\hat\theta_j)\ \big)=1-\alpha.
$$
Replace $c = t_{1-\alpha/2,\nu}$ and plug in the standard error to get the familiar (two-sided) CI:
$$
\boxed{
\hat{\theta}_j \pm t_{1-\alpha/2,\nu}\; \hat{\sigma}\; \sqrt{\bigl[(\hat{\mathbf X}^\top\hat{\mathbf X})^{-1}\bigr]_{jj}}\quad\blacksquare
}
$$

__Wow!__ Mind blown! Ok, so what about one-sided intervals? Same approach for one-sided intervals; start from $\Pr(T_j \le t_{1-\alpha,\nu})=1-\alpha$ (or the lower tail) and solve for $\theta_j$. The inversion step just converts bounds on the statistic into bounds on the parameter via a monotone transformation.

___

## Lab
In [L7d](../L7d/CHEME-5800-L7d-Lab-HousingPriceOLS-Fall-2026.ipynb), we will build a linear model of housing prices and estimate its parameters with ordinary least squares.

___

## Summary
We derived the ordinary least squares estimate for overdetermined and underdetermined data matrices, wrote it in terms of the singular value decomposition, and quantified the uncertainty in the estimated parameters.

> __Key Takeaways:__
>
> * __Overdetermined and underdetermined problems:__ With more observations than parameters, we minimized the sum of squared residuals and found an unbiased estimate whose fitted values are the projection of the observations onto the column space of the data matrix. With fewer observations than parameters, infinitely many parameter vectors fit the data exactly, and we chose the one with the smallest norm.
>
> * __Least squares with the SVD:__ We wrote the least-squares estimate as a weighted sum of the right singular vectors, with each weight divided by its singular value, so small singular values amplify the noise in the data. This form handles rank-deficient data matrices and avoids multiplying the transpose of the data matrix by the data matrix, which squares the condition number.
>
> * __Parameter uncertainty:__ We estimated the error variance from the residuals and propagated it through the OLS estimate to get the variance of each estimated parameter. The square roots of these variances are the standard errors, which give confidence intervals and hypothesis tests from Student's t-distribution.

In week 8, we add a penalty to the least-squares objective (regularization) and check how well the fitted models generalize to new data.

___